# Tarea Práctica - Módulo 6 PySpark

**Sistemas de Computación Distribuida - FAE Usach**
**Profesor: Ivan Espinoza**

---

## Datos del alumno

| Campo | Valor |
|-------|-------|
| Nombre completo | Carlos |
| Apellido | Salinas |
| RUT | 19583099-1 |
| Email | carlossalinasmorales1@gmail.com |
| Pareja (si aplica) | No |
| Fecha entrega | _completa aquí_ |

---

## Instrucciones generales

1. Esta plantilla tiene el **setup ya armado** (descarga del dataset y SparkSession). No necesitas modificarla.
2. Cada ejercicio tiene celdas con `# TODO:` donde debes escribir tu código.
3. Antes de entregar:
   - Ejecuta **todo el notebook desde cero** (Runtime > Restart and run all).
   - Verifica que **todas las celdas se ejecuten sin errores**.
   - Verifica que **al final no quede ningun stream activo**.
4. Guarda el notebook con el nombre `Tarea_M6_<apellido>_<nombre>.ipynb` y envíalo por correo a ivan.espinoza.m@gmail.com.


---
## Setup (no modificar)

### Instalación de PySpark


In [29]:
#!pip install pyspark==3.5.1 --quiet
#print("PySpark instalado")

### Crear SparkSession

In [30]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("Tarea_M6_FAE_USACH")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.sql.streaming.statefulOperator.checkCorrectness.enabled", "false")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")
print("SparkSession lista. Versión:", spark.version)

SparkSession lista. Versión: 3.5.1


### Descargar el dataset MovieLens

In [31]:
#Descargas para Linux (Para windows decargar manualmente y colocar en carpeta data)

# # Descargamos y descomprimimos el dataset (~1 MB)
# !wget -q https://files.grouplens.org/datasets/movielens/ml-latest-small.zip -O /tmp/ml.zip
# !unzip -q -o /tmp/ml.zip -d /tmp/
# print("Archivos disponibles:")
# !ls /tmp/ml-latest-small/

**Rutas de los archivos** (úsalas en los ejercicios):

```python
PATH_MOVIES  = "/tmp/ml-latest-small/movies.csv"
PATH_RATINGS = "/tmp/ml-latest-small/ratings.csv"
PATH_TAGS    = "/tmp/ml-latest-small/tags.csv"
PATH_LINKS   = "/tmp/ml-latest-small/links.csv"
```


In [32]:
#PATHS PARA LINUX

# PATH_MOVIES  = "/tmp/ml-latest-small/movies.csv"
# PATH_RATINGS = "/tmp/ml-latest-small/ratings.csv"
# PATH_TAGS    = "/tmp/ml-latest-small/tags.csv"
# PATH_LINKS   = "/tmp/ml-latest-small/links.csv"

print("Variables definidas")

Variables definidas


In [33]:
#PATHS PARA WINDOWS

PATH_MOVIES  = "./data/movies.csv"
PATH_RATINGS = "./data/ratings.csv"
PATH_TAGS    = "./data/tags.csv"
PATH_LINKS   = "./data/links.csv"

print("Variables definidas")

Variables definidas


---
# PARTE A - Fundamentos (Clase 2)

## Ejercicio 1 - Carga y exploración (10 pts)

Carga los archivos `movies.csv` y `ratings.csv` con **schema explícito**. Verifica:
- `printSchema` de ambos.
- `count()` de ambos.
- `show(5)` de ambos.

**Tipos sugeridos:**
- movies: `movieId` (int), `title` (string), `genres` (string).
- ratings: `userId` (int), `movieId` (int), `rating` (double), `timestamp` (long).

**Esperado:** movies ~9.700 filas, ratings ~100.000 filas.


In [34]:
#Exploracion de los df raw

df_movies = spark.read.csv(PATH_MOVIES, header=True)
df_ratings = spark.read.csv(PATH_RATINGS, header=True)

print("*************** Movies info ***************")
df_movies.show(5)
print(f"Cantidad de filas: {df_movies.count()}")
print(f"Schema raw de movies:")
df_movies.printSchema()


print("*************** Ratings info ********************")
df_ratings.show(5)
print(f"Number de filas: {df_ratings.count()}")
print(f"Schema raw de ratings:")
df_ratings.printSchema()

*************** Movies info ***************
+-------+--------------------+--------------------+
|movieId|               title|              genres|
+-------+--------------------+--------------------+
|      1|    Toy Story (1995)|Adventure|Animati...|
|      2|      Jumanji (1995)|Adventure|Childre...|
|      3|Grumpier Old Men ...|      Comedy|Romance|
|      4|Waiting to Exhale...|Comedy|Drama|Romance|
|      5|Father of the Bri...|              Comedy|
+-------+--------------------+--------------------+
only showing top 5 rows

Cantidad de filas: 9742
Schema raw de movies:
root
 |-- movieId: string (nullable = true)
 |-- title: string (nullable = true)
 |-- genres: string (nullable = true)

*************** Ratings info ********************
+------+-------+------+---------+
|userId|movieId|rating|timestamp|
+------+-------+------+---------+
|     1|      1|   4.0|964982703|
|     1|      3|   4.0|964981247|
|     1|      6|   4.0|964982224|
|     1|     47|   5.0|964983815|
|     1| 

In [35]:
#Crear y enlazar schemas

from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DoubleType, LongType

schema_movies = StructType([
    StructField("movieId", IntegerType(), False),
    StructField("title", StringType(), nullable=False),
    StructField("genres", StringType(), nullable=True)
])

schema_ratings = StructType([
    StructField("userId", IntegerType(), nullable=False),
    StructField("movieId", IntegerType(), nullable=False),
    StructField("rating", DoubleType(), nullable=False),
    StructField("timestamp", LongType(), nullable=False)
])


df_movies = (
    spark.read.csv(PATH_MOVIES, header=True, schema=schema_movies)
)

df_ratings = (
    spark.read.csv(PATH_RATINGS, header=True, schema=schema_ratings)
)

print("*************** Movies info ***************")
df_movies.show(5)
print(f"Cantidad de filas: {df_movies.count()}")
print(f"Schema raw de movies:")
df_movies.printSchema()


print("*************** Ratings info ********************")
df_ratings.show(5)
print(f"Number de filas: {df_ratings.count()}")
print(f"Schema raw de ratings:")
df_ratings.printSchema()




*************** Movies info ***************
+-------+--------------------+--------------------+
|movieId|               title|              genres|
+-------+--------------------+--------------------+
|      1|    Toy Story (1995)|Adventure|Animati...|
|      2|      Jumanji (1995)|Adventure|Childre...|
|      3|Grumpier Old Men ...|      Comedy|Romance|
|      4|Waiting to Exhale...|Comedy|Drama|Romance|
|      5|Father of the Bri...|              Comedy|
+-------+--------------------+--------------------+
only showing top 5 rows

Cantidad de filas: 9742
Schema raw de movies:
root
 |-- movieId: integer (nullable = true)
 |-- title: string (nullable = true)
 |-- genres: string (nullable = true)

*************** Ratings info ********************
+------+-------+------+---------+
|userId|movieId|rating|timestamp|
+------+-------+------+---------+
|     1|      1|   4.0|964982703|
|     1|      3|   4.0|964981247|
|     1|      6|   4.0|964982224|
|     1|     47|   5.0|964983815|
|     1|

## Ejercicio 2 - Transformaciones básicas (15 pts)

### Parte a) y b) - Fechas en ratings

- Convierte `timestamp` (epoch seconds) a un `TimestampType`. Llama a la columna **`fecha`**.
- Agrega una columna **`anio_rating`** con el año del rating.

### Parte c) y d) - Limpieza de títulos en movies

- Crea **`titulo_limpio`**: el título sin el año ni los paréntesis. Ej: 'Toy Story'.
- Crea **`anio_pelicula`**: el año de la película como **integer**. Ej: 1995.
- Muestra el top 10 películas con el **titulo_limpio más largo** (usa `length`).


In [36]:
from pyspark.sql.functions import col, from_unixtime, year, regexp_extract, regexp_replace, length

# parte a y b - agregar columna fecha y anio_rating a df_ratings
df_ratings = df_ratings.withColumn("fecha", from_unixtime(col("timestamp"))) \
                       .withColumn("anio_rating", year(col("fecha")))

print("*************** Ratings info con fecha y año de rating ********************")
df_ratings.show(5)

# parte c - agregar titulo_limpio y anio_pelicula a df_movies
df_movies = df_movies.withColumn("titulo_limpio", regexp_replace(col("title"), r"\s*\(\d{4}\)\s*$", "")) \
                     .withColumn("anio_pelicula", regexp_extract(col("title"), r"\((\d{4})\)", 1).cast("int"))

print("*************** Movies info con titulo limpio y año de película ********************")
df_movies.show(5)

# parte d - top 10 con titulo_limpio más largo
df_top_10_long_title = df_movies.withColumn("longitud_titulo", length(col("titulo_limpio")))
top_10_longitud = df_top_10_long_title.orderBy(col("longitud_titulo").desc()).limit(10)

print("*************** Top 10 títulos limpios más largos ********************")
top_10_longitud.select("titulo_limpio", "longitud_titulo").show(truncate=False)



*************** Ratings info con fecha y año de rating ********************
+------+-------+------+---------+-------------------+-----------+
|userId|movieId|rating|timestamp|              fecha|anio_rating|
+------+-------+------+---------+-------------------+-----------+
|     1|      1|   4.0|964982703|2000-07-30 14:45:03|       2000|
|     1|      3|   4.0|964981247|2000-07-30 14:20:47|       2000|
|     1|      6|   4.0|964982224|2000-07-30 14:37:04|       2000|
|     1|     47|   5.0|964983815|2000-07-30 15:03:35|       2000|
|     1|     50|   5.0|964982931|2000-07-30 14:48:51|       2000|
+------+-------+------+---------+-------------------+-----------+
only showing top 5 rows

*************** Movies info con titulo limpio y año de película ********************
+-------+--------------------+--------------------+--------------------+-------------+
|movieId|               title|              genres|       titulo_limpio|anio_pelicula|
+-------+--------------------+----------------

## Ejercicio 3 - Agregaciones simples (15 pts)

Calcula por película:
- cantidad de ratings (count)
- rating promedio (avg)
- rating mínimo y máximo

Filtra solo películas con **al menos 50 ratings** (así los promedios son confiables).

Muestra el **top 10 mejor evaluadas** (con su título limpio del Ejercicio 2).


In [37]:
from pyspark.sql.functions import avg, count, min as f_min, max as f_max, round as f_round

# agregar por movieId
df_ratings_stats = df_movies.join(df_ratings, "movieId", "inner") \
                    .groupBy("movieId") \
                    .agg(count("rating").alias("cantidad_ratings"),
                         f_round(avg("rating"), 2).alias("promedio_rating"),
                         f_min("rating").alias("min_rating"),
                         f_max("rating").alias("max_rating"))

print("*************** df con calculo de ratings ***************")
df_ratings_stats.show(5)
print(df_ratings_stats.count())



# filtrar cantidad >= 50
df_ratings_mas_de_50 = df_ratings_stats.filter(col("cantidad_ratings") >= 50)

print("*************** df filtrado por cantidad_ratings >= 50 ***************")
df_ratings_mas_de_50.show(5)
print(df_ratings_mas_de_50.count())


# hacer join con movies para tener el titulo_limpio
df_ratings_mas_de_50_con_titulo_limpio = df_ratings_mas_de_50.join(df_movies, "movieId", "left").select("movieId", "titulo_limpio", "genres", "cantidad_ratings", "promedio_rating", "min_rating", "max_rating")

print("*************** df con ratings mayores a 50 y titulo_limpio ***************")
df_ratings_mas_de_50_con_titulo_limpio.show(5)

# TODO: mostrar top 10
print("*************** top 10 películas con mejor promedio de ratings ***************")
df_ratings_mas_de_50_con_titulo_limpio.orderBy(col("promedio_rating").desc()).limit(10).select("titulo_limpio", "promedio_rating", "cantidad_ratings").show(truncate=False)



*************** df con calculo de ratings ***************
+-------+----------------+---------------+----------+----------+
|movieId|cantidad_ratings|promedio_rating|min_rating|max_rating|
+-------+----------------+---------------+----------+----------+
|     70|              55|           3.51|       1.0|       5.0|
|    157|              11|           2.86|       1.5|       5.0|
|    362|              34|           3.53|       1.0|       5.0|
|    457|             190|           3.99|       0.5|       5.0|
|    673|              53|           2.71|       0.5|       5.0|
+-------+----------------+---------------+----------+----------+
only showing top 5 rows

9724
*************** df filtrado por cantidad_ratings >= 50 ***************
+-------+----------------+---------------+----------+----------+
|movieId|cantidad_ratings|promedio_rating|min_rating|max_rating|
+-------+----------------+---------------+----------+----------+
|     70|              55|           3.51|       1.0|       5

---
# PARTE B - Joins y transformaciones avanzadas (Clase 3)

## Ejercicio 4 - Análisis de géneros (15 pts)

La columna `genres` viene como string con géneros separados por `|`. Una película puede tener varios géneros.

- **a)** Crea `df_peli_genero` con UNA fila por (película, género). Tip: `split` + `explode`.
- **b)** Top 10 géneros por cantidad de películas.
- **c)** Une `df_peli_genero` con ratings, calcula:
  - ratings totales por género
  - rating promedio por género
- **d)** Top 5 géneros con mejor promedio, **considerando solo géneros con > 1.000 ratings totales**.


In [46]:
from pyspark.sql.functions import split, explode, when

print("*************** df movies clean ***************")
df_movies_clean = df_movies.selectExpr("movieId", "titulo_limpio AS titulo", "genres AS generos", "anio_pelicula AS anio_estreno")
df_movies_clean.show(5)



# parte a - df_peli_genero (una fila por película y género)
df_peli_genero = df_movies_clean.select("movieId", "titulo", explode(split(col("generos"), "\\|")).alias("genero"), "anio_estreno")

print("*************** df con una fila por película y género ***************")
df_peli_genero.show(5)



# parte b - top 10 géneros por cantidad de películas
df_top_generos = df_peli_genero.groupBy("genero").agg(count("movieId").alias("cantidad_peliculas"))

print("*************** top 10 géneros por cantidad de películas ***************")
df_top_generos.orderBy(col("cantidad_peliculas").desc()).limit(10).show(truncate=False)



# parte c - join con ratings y agregaciones por género
# **** Asumo que si una pelicula aporta varios ratings para distintos generos, se cuenta para cada uno de ellos **********
df_peli_genero_ratings = df_peli_genero.join(df_ratings_stats, "movieId", "inner").select("genero", "cantidad_ratings", "promedio_rating")
df_ratings_totales_y_promedio_por_genero = df_peli_genero_ratings.groupBy("genero") \
                                                            .agg(count("cantidad_ratings").alias("cantidad_ratings"),
                                                                 f_round(avg("promedio_rating"), 2).alias("promedio_rating"))

print("*************** df con estadísticas por género ***************")
df_ratings_totales_y_promedio_por_genero.show(5)



# parte d - top 5 géneros con mejor promedio (> 1000 ratings)
df_top_5_generos_rating = df_ratings_totales_y_promedio_por_genero.filter(col("cantidad_ratings") > 1000) \
                                        .orderBy(col("promedio_rating").desc()) \
                                        .limit(5)

print("*************** top 5 géneros con mejor promedio (> 1000 ratings) ***************")
df_top_5_generos_rating.show(truncate=False)


*************** df movies clean ***************
+-------+--------------------+--------------------+------------+
|movieId|              titulo|             generos|anio_estreno|
+-------+--------------------+--------------------+------------+
|      1|           Toy Story|Adventure|Animati...|        1995|
|      2|             Jumanji|Adventure|Childre...|        1995|
|      3|    Grumpier Old Men|      Comedy|Romance|        1995|
|      4|   Waiting to Exhale|Comedy|Drama|Romance|        1995|
|      5|Father of the Bri...|              Comedy|        1995|
+-------+--------------------+--------------------+------------+
only showing top 5 rows

*************** df con una fila por película y género ***************
+-------+---------+---------+------------+
|movieId|   titulo|   genero|anio_estreno|
+-------+---------+---------+------------+
|      1|Toy Story|Adventure|        1995|
|      1|Toy Story|Animation|        1995|
|      1|Toy Story| Children|        1995|
|      1|Toy S

## Ejercicio 5 - Window functions y escritura particionada (20 pts)

### Parte a) - Top 3 películas por género

Para cada género, ranking de las **3 películas con mejor rating promedio** (entre las que tengan al menos 30 ratings).

Esperado: una tabla con `género`, `titulo_limpio`, `promedio`, `posicion` (1, 2 o 3). 3 filas por género.

### Parte b) - Evolución temporal

Rating promedio agrupado por (`anio_pelicula`, `género`). Filtra películas desde 1980 en adelante.

### Parte c) - Escritura particionada

Escribe el resultado de (b) particionado por `género` en Parquet en `/tmp/ratings_por_genero/`. Verifica con `ls`.


In [60]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number

# TODO: parte a - ranking por género (top 3)
df_peli_genero_ratings = df_peli_genero.join(df_ratings_stats, "movieId", "inner").select("movieId", "titulo", "genero", "cantidad_ratings", "promedio_rating", "anio_estreno")
df_peli_genero_ratings_filtrado = df_peli_genero_ratings.filter(col("cantidad_ratings") >= 30).select("movieId", "titulo", "genero", "cantidad_ratings", "promedio_rating")

print("*************** df con ratings por película y género filtrado por cantidad_ratings >= 30 ***************")
df_peli_genero_ratings_filtrado.show(5)

window_rank_por_genero = Window.partitionBy("genero").orderBy(col("promedio_rating").desc())

df_ranking_genero = df_peli_genero_ratings_filtrado.withColumn("rank", row_number().over(window_rank_por_genero))

print("*************** top 3 películas por género con al menos 30 ratings ***************")
df_ranking_genero.select("genero", "titulo", "promedio_rating", "cantidad_ratings", "rank").filter(col("rank") <= 3).orderBy("genero", "rank").show(truncate=False)

*************** df con ratings por película y género filtrado por cantidad_ratings >= 30 ***************
+-------+-------------------+--------+----------------+---------------+
|movieId|             titulo|  genero|cantidad_ratings|promedio_rating|
+-------+-------------------+--------+----------------+---------------+
|     70|From Dusk Till Dawn|Thriller|              55|           3.51|
|     70|From Dusk Till Dawn|  Horror|              55|           3.51|
|     70|From Dusk Till Dawn|  Comedy|              55|           3.51|
|     70|From Dusk Till Dawn|  Action|              55|           3.51|
|    362|   Jungle Book, The| Romance|              34|           3.53|
+-------+-------------------+--------+----------------+---------------+
only showing top 5 rows

*************** top 3 películas por género con al menos 30 ratings ***************
+-----------+--------------------------------------------------------------------+---------------+----------------+----+
|genero     |titul

In [74]:
# TODO: parte b - evolución temporal (anio_pelicula, género) desde 1980

#Ocupamos la df generada en el bloque anterior
print("*************** df con ratings por película, género y anio ***************")
print(df_peli_genero_ratings.show(5))

df_evolucion = df_peli_genero_ratings.filter((col("anio_estreno") >= 1980) & (col("genero") != "(no genres listed)")) \
                                     .groupBy("anio_estreno", "genero") \
                                     .agg(count("movieId").alias("cantidad_peliculas"),
                                          f_round(avg("promedio_rating"), 2)) \
                                     .orderBy("genero", "anio_estreno") # decidí hacer el filtro primero por genero, asi en al tabla se ve la evolucion de cada genero mas facilmente

print("*************** evolución temporal de películas por género desde 1980 ***************")
df_evolucion.show(50, truncate=False)


*************** df con ratings por película, género y anio ***************
+-------+-------------------+--------+----------------+---------------+------------+
|movieId|             titulo|  genero|cantidad_ratings|promedio_rating|anio_estreno|
+-------+-------------------+--------+----------------+---------------+------------+
|     70|From Dusk Till Dawn|Thriller|              55|           3.51|        1996|
|     70|From Dusk Till Dawn|  Horror|              55|           3.51|        1996|
|     70|From Dusk Till Dawn|  Comedy|              55|           3.51|        1996|
|     70|From Dusk Till Dawn|  Action|              55|           3.51|        1996|
|    157|     Canadian Bacon|     War|              11|           2.86|        1995|
+-------+-------------------+--------+----------------+---------------+------------+
only showing top 5 rows

None
*************** evolución temporal de películas por género desde 1980 ***************
+------------+---------+------------------+-

In [81]:
import pyspark.sql.functions as F

print("*************** df con ratings por película, género y anio ***************")
print(df_peli_genero_ratings.show(5))

# 1. Definimos la ventana particionada por año y género
# Nota: No lleva .orderBy() porque queremos que la agregación (count y avg) 
# se aplique a todo el grupo completo, tal como lo hace el groupBy.
window_evolucion = Window.partitionBy("anio_estreno", "genero")

# 2. Aplicamos las funciones de ventana y eliminamos los duplicados
df_evolucion = df_peli_genero_ratings.filter((F.col("anio_estreno") >= 1980) & (F.col("genero") != "(no genres listed)")) \
    .withColumn("cantidad_peliculas", F.count("movieId").over(window_evolucion)) \
    .withColumn("promedio_rating", F.round(F.avg("promedio_rating").over(window_evolucion), 2)) \
    .select("anio_estreno", "genero", "cantidad_peliculas", "promedio_rating") \
    .dropDuplicates(["anio_estreno", "genero"]) \
    .orderBy("genero", "anio_estreno")

print("*************** evolución temporal de películas por género desde 1980 ***************")
df_evolucion.show(50, truncate=False)

*************** df con ratings por película, género y anio ***************
+-------+-------------------+--------+----------------+---------------+------------+
|movieId|             titulo|  genero|cantidad_ratings|promedio_rating|anio_estreno|
+-------+-------------------+--------+----------------+---------------+------------+
|     70|From Dusk Till Dawn|Thriller|              55|           3.51|        1996|
|     70|From Dusk Till Dawn|  Horror|              55|           3.51|        1996|
|     70|From Dusk Till Dawn|  Comedy|              55|           3.51|        1996|
|     70|From Dusk Till Dawn|  Action|              55|           3.51|        1996|
|    157|     Canadian Bacon|     War|              11|           2.86|        1995|
+-------+-------------------+--------+----------------+---------------+------------+
only showing top 5 rows

None
*************** evolución temporal de películas por género desde 1980 ***************
+------------+---------+------------------+-

In [82]:
# TODO: parte c - escribir particionado por género
# df_evolucion.write.mode('overwrite').partitionBy('genero').parquet('./data/output/ratings_por_genero') #Este seria el comando para windows pero no funciona (con mi config actual)
try :
    df_evolucion.write.mode('overwrite').partitionBy('genero').parquet('/tmp/ratings_por_genero') # Este si funciona pero solo linux y colab
except Exception as e:
    print("Error al escribir el dataframe revisa carpetas de guardado o configuracion de sistema:", e)

# Verificar
# !ls /tmp/ratings_por_genero/ | head -10



Error al escribir el dataframe revisa carpetas de guardado o configuracion de sistema: An error occurred while calling o1945.parquet.
: java.lang.RuntimeException: java.io.FileNotFoundException: java.io.FileNotFoundException: Hadoop home directory C:\hadoop does not exist -see https://wiki.apache.org/hadoop/WindowsProblems
	at org.apache.hadoop.util.Shell.getWinUtilsPath(Shell.java:735)
	at org.apache.hadoop.util.Shell.getSetPermissionCommand(Shell.java:270)
	at org.apache.hadoop.util.Shell.getSetPermissionCommand(Shell.java:286)
	at org.apache.hadoop.fs.RawLocalFileSystem.setPermission(RawLocalFileSystem.java:978)
	at org.apache.hadoop.fs.RawLocalFileSystem.mkOneDirWithMode(RawLocalFileSystem.java:660)
	at org.apache.hadoop.fs.RawLocalFileSystem.mkdirsWithOptionalPermission(RawLocalFileSystem.java:700)
	at org.apache.hadoop.fs.RawLocalFileSystem.mkdirs(RawLocalFileSystem.java:672)
	at org.apache.hadoop.fs.RawLocalFileSystem.mkdirsWithOptionalPermission(RawLocalFileSystem.java:699)
	at

---
# PARTE C - Streaming (Clase 4)

## Ejercicio 6 - Simulacion de stream de ratings (20 pts)

### Estrategia

1. Particionamos `ratings` por `anio_rating` en archivos JSONL.
2. Un thread los copia uno por uno a una carpeta destino con pausas (simulando que llegan en vivo).
3. Un `readStream` los procesa.

### Pasos

**a)** Particiona ratings en archivos JSONL por año. Cada archivo: `/tmp/ratings_por_anio/ratings_AAAA.jsonl`.

**b)** Define un productor que copia los archivos uno a uno hacia `/tmp/ratings_stream/` con pausas de 3 segundos.

**c)** Lanza un `readStream` sobre `/tmp/ratings_stream/` con schema explícito.

**d)** Calcula en vivo el **rating promedio y cantidad por película**. Output mode `update`. Memory sink.

**e)** Después de 30 segundos, consulta y muestra el top 10 películas por cantidad de ratings acumulados. Detener el stream.


In [ ]:
import os, shutil, json, time, threading
import pandas as pd

# Limpieza
for d in ["/tmp/ratings_por_anio", "/tmp/ratings_stream"]:
    if os.path.exists(d):
        shutil.rmtree(d)
os.makedirs("/tmp/ratings_por_anio", exist_ok=True)
os.makedirs("/tmp/ratings_stream", exist_ok=True)
print("Carpetas limpias")

ModuleNotFoundError: No module named 'pandas'

In [ ]:
# TODO: parte a) particionar ratings por anio_rating
# Para cada año distinto, escribir un JSONL en /tmp/ratings_por_anio/ratings_AAAA.jsonl
# Tip: puedes hacer .toPandas() y usar pandas.to_json(..., orient='records', lines=True)
# (es OK porque el dataset es chico, ~100k filas)

# anios_unicos = sorted([r[0] for r in df_ratings.select('anio_rating').distinct().collect()])
# for anio in anios_unicos:
#     ...



In [ ]:
# TODO: parte b) productor que copia archivos uno a uno

# def productor(...):
#     for archivo in sorted(os.listdir('/tmp/ratings_por_anio')):
#         shutil.copy(...)
#         time.sleep(3)



In [ ]:
# TODO: parte c) definir el readStream
from pyspark.sql.types import StructType, StructField, IntegerType, DoubleType, LongType, StringType, TimestampType

# schema_stream = StructType([...])

# df_stream = (
#     spark.readStream
#     .schema(schema_stream)
#     .option("maxFilesPerTrigger", 1)
#     .json("/tmp/ratings_stream")
# )



In [ ]:
# TODO: parte d) calcular agregado y lanzar a memoria

# df_agg = df_stream.groupBy('movieId').agg(...)

# query = (
#     df_agg.writeStream
#     .format("memory")
#     .queryName("ratings_vivo")
#     .outputMode("update")
#     .start()
# )



In [ ]:
# TODO: parte e) lanzar el productor en thread, esperar 30s, consultar, detener

# t = threading.Thread(target=productor, daemon=True)
# t.start()
# time.sleep(30)
# spark.sql("SELECT * FROM ratings_vivo ORDER BY cantidad DESC LIMIT 10").show()
# query.stop()



---
# Bonus opcional (+10 pts)

Elige **una** de las dos opciones (no acumulables):

## Opcion A - Tag analysis

Lee `tags.csv` y encuentra:
- Top 10 tags más frecuentes.
- Para esas 10 tags: rating promedio de las películas asociadas.

## Opcion B - Stream con ventana temporal

En el Ejercicio 6, en vez del agregado global, calcula **rating promedio por ventana tumbling de 5 segundos**. Muestra las últimas 5 ventanas.


In [ ]:
# Opcion A 
df_tags = spark.read.csv(PATH_TAGS, header=True, schema=StructType([
    StructField("userId", IntegerType(), nullable=False),
    StructField("movieId", IntegerType(), nullable=False),
    StructField("tag", StringType(), nullable=False),
    StructField("timestamp", LongType(), nullable=False)
]))

df_tags.printSchema()
df_tags.show(5)

df_10_tags_mas_comunes = df_tags.groupBy("tag").agg(count("tag").alias("cantidad")).orderBy(col("cantidad").desc()).limit(10)

print("*************** top 10 tags más comunes ***************")
df_10_tags_mas_comunes.show(truncate=False)


#FALTA TERMINAR


root
 |-- userId: integer (nullable = true)
 |-- movieId: integer (nullable = true)
 |-- tag: string (nullable = true)
 |-- timestamp: long (nullable = true)

+------+-------+---------------+----------+
|userId|movieId|            tag| timestamp|
+------+-------+---------------+----------+
|     2|  60756|          funny|1445714994|
|     2|  60756|Highly quotable|1445714996|
|     2|  60756|   will ferrell|1445714992|
|     2|  89774|   Boxing story|1445715207|
|     2|  89774|            MMA|1445715200|
+------+-------+---------------+----------+
only showing top 5 rows

*************** top 10 tags más comunes ***************
+-----------------+--------+
|tag              |cantidad|
+-----------------+--------+
|In Netflix queue |131     |
|atmospheric      |36      |
|thought-provoking|24      |
|superhero        |24      |
|surreal          |23      |
|Disney           |23      |
|funny            |23      |
|religion         |22      |
|psychology       |21      |
|quirky         

---
# Cierre - detener todo y cerrar SparkSession

Esta celda es obligatoria. Si la dejas fuera te resto puntos por dejar streams activos.


In [ ]:
# Detener todos los streams activos por las dudas
queries_activas = [q for q in spark.streams.active]
print(f"Queries activas a detener: {len(queries_activas)}")
for q in queries_activas:
    q.stop()
    print(f"  Detenida: {q.id}")

spark.stop()
print("SparkSession cerrada. Tarea terminada.")